In [1]:
import pandas as pd

In [2]:
FILE_COUNTS = "TCGA-KIRC.star_counts.tsv"
FILE_CLINICAL = "TCGA-KIRC.clinical.tsv"

In [3]:
#LOADING

df_counts = pd.read_csv(FILE_COUNTS, sep='\t', index_col=0)
df_clinical = pd.read_csv(FILE_CLINICAL, sep='\t', index_col=0)

In [4]:
#TABLE TRANSPOSITION

df_counts = df_counts.T

In [5]:
#CHANGING LOGARYTMIC SCALE TO RAW INTEGREG COUNTS - UCSC XENA NEED

check_val = df_counts.iloc[0, 0]
if isinstance(check_val, float) and not check_val.is_integer():
    print(f"The data is logarithmic: (value: {check_val:.2f}) -logarithm inversion")
    df_counts = (2 ** df_counts) - 1
    df_counts = df_counts.round().astype(int)
    print("Data changed to raw counts.")
else:
    print("The data are already integers..")

The data is logarithmic: (value: 11.64) -logarithm inversion
Data changed to raw counts.


In [6]:
#TYPE TEST

check_val = df_counts.iloc[0, 0]
print(f"DEBUG: Sample value in data: {check_val} (Typ: {type(check_val)})")

DEBUG: Sample value in data: 3195 (Typ: <class 'numpy.int64'>)


In [7]:
#Patient selection based on clinical labels

target_col = 'sample_type.samples'
kept_samples = df_clinical[
    df_clinical[target_col].isin(['Primary Tumor', 'Solid Tissue Normal'])
].index

In [8]:
#COMMON PART - ONLY THOSE IDs THAT ARE IN BOTH TABLES - CLINICAL AND GENES

common_samples = df_counts.index.intersection(kept_samples)
print(f"Number of patients for analysis: {len(common_samples)}")

Number of patients for analysis: 609


In [9]:
#CLASS STATISTICS

y_temp = df_clinical.loc[common_samples, target_col]
print("\nCLASS STATISTICS:")
print(y_temp.value_counts())


CLASS STATISTICS:
sample_type.samples
Primary Tumor          537
Solid Tissue Normal     72
Name: count, dtype: int64


In [10]:
#PREPARATION X AND Y

X = df_counts.loc[common_samples]
y = y_temp.map({'Solid Tissue Normal': 0, 'Primary Tumor': 1})

In [11]:
#SAVE TO CSV

X.to_csv("KIRC_raw_counts.csv")
y.to_csv("KIRC_labels.csv")

In [12]:
#FINAL NUMBERS

df = pd.read_csv("KIRC_raw_counts.csv", index_col=0)
print(f"\nNumber of patients: {df.shape[0]}")
print(f"Number of genes: {df.shape[1]}")


Number of patients: 609
Number of genes: 60660
